<a href="https://colab.research.google.com/github/tharujayasinghe163/Statistical-Learning-e22163/blob/main/Assignment%207d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Bayesian Estimation for Structural Health Monitoring (SHM) via Bounded Grid Updates**.

---

## 1. Problem Formulation & System Model

In Structural Health Monitoring (SHM), we aim to estimate unknown structural health parameters (e.g., stiffness reduction factor, crack depth, or loss of cross-sectional area) denoted by $\theta \in [\theta_{\min}, \theta_{\max}] \subset \mathbb{R}^d$ given noisy sensor measurements $y^{(k)} = (y_1, y_2, \dots, y_k)$.

### Measurement Equation

At step $k$, a physical or surrogate model $h_k(\cdot)$ maps the structural parameter $\theta$ to the expected sensor response $\hat{y}_k$:


$$y_k = h_k(\theta) + \varepsilon_k, \quad \varepsilon_k \sim \mathcal{N}(0, \sigma_k^2)$$

where:

* $y_k \in \mathbb{R}$ is the observed sensor data (e.g., strain, acceleration, modal frequency shift) at time/step $k$.
* $h_k(\theta)$ is the forward model output evaluated at parameter state $\theta$.
* $\varepsilon_k$ is the Gaussian measurement noise with variance $\sigma_k^2$.

---

## 2. Mathematical Formulation of Sequential Grid Updates

### 1. Likelihood Function

The likelihood contribution $L(y_k \mid \theta)$ of observing measurement $y_k$ given structural parameter $\theta$ is:


$$L(y_k \mid \theta) = \frac{1}{\sqrt{2\pi \sigma_k^2}} \exp\left( -\frac{\left(y_k - h_k(\theta)\right)^2}{2\sigma_k^2} \right)$$

Assuming independent measurement errors conditional on $\theta$, the joint likelihood over history vector $y^{(k)} = (y_1, \dots, y_k)$ is:


$$L(y^{(k)} \mid \theta) = \prod_{i=1}^k \frac{1}{\sqrt{2\pi \sigma_i^2}} \exp\left( -\frac{\left(y_i - h_i(\theta)\right)^2}{2\sigma_i^2} \right)$$

---

### 2. Recursive Posterior Update

Using the posterior from step $k-1$ as the prior for step $k$, the posterior probability density function $f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)})$ satisfies:

$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) = \frac{L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})}{\int_{\theta_{\min}}^{\theta_{\max}} L(y_k \mid s) \cdot f_{\Theta \mid Y^{(k-1)}}(s \mid y^{(k-1)}) \, ds}$$

Up to a proportionality constant:


$$f_{\Theta \mid Y^{(k)}}(\theta \mid y^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid Y^{(k-1)}}(\theta \mid y^{(k-1)})$$

---

## 3. Bounded Fixed-Grid Discretization Scheme

When closed-form updates are intractable due to nonlinear forward physics $h_k(\theta)$, a **fixed grid numerical integration** approach over the bounded domain $[\theta_{\min}, \theta_{\max}]$ provides a deterministic solution:

### Step-by-Step Algorithm

1. **Domain Discretization:** Define $M$ discrete evaluation points across the physical bounds:

$$\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M], \quad \theta_1 = \theta_{\min}, \, \theta_M = \theta_{\max}$$


2. **Prior Initialization:** Initialize the prior density vector $\mathbf{f}^{(0)}$ over the grid (e.g., Uniform or Gaussian truncated to $[\theta_{\min}, \theta_{\max}]$):

$$\mathbf{f}^{(0)}_m = \frac{1}{\theta_{\max} - \theta_{\min}} \quad \text{for } m = 1, \dots, M$$


3. **Sequential Measurement Evaluation (At Step $k$):**
* Evaluate model outputs: $\mathbf{h}_k = [h_k(\theta_1), h_k(\theta_2), \dots, h_k(\theta_M)]^\top$.
* Evaluate likelihood vector $\mathbf{L}_k$:

$$\mathbf{L}_{k, m} = \exp\left( -\frac{(y_k - h_k(\theta_m))^2}{2\sigma_k^2} \right)$$


* Calculate unnormalized posterior vector:

$$\mathbf{q}^{(k)} = \mathbf{L}_k \odot \mathbf{f}^{(k-1)}$$


* Compute normalizing factor via trapezoidal integration:

$$I_k = \int_{\theta_{\min}}^{\theta_{\max}} q^{(k)}(s) \, ds \approx \text{trapezoid}(\mathbf{q}^{(k)}, \boldsymbol{\theta})$$


* Update normalized posterior:

$$\mathbf{f}^{(k)} = \frac{\mathbf{q}^{(k)}}{I_k}$$





---

## 4. Key Bayesian Estimators

At any step $k$, point estimates and uncertainty bounds are computed directly from the discrete grid density $\mathbf{f}^{(k)}$:

| Estimator / Metric | Mathematical Formula | Numerical Discrete Computation |
| --- | --- | --- |
| **Bayes Estimate (Posterior Mean)** | $\hat{\theta}_{\text{Bayes}}^{(k)} = \int_{\theta_{\min}}^{\theta_{\max}} \theta f(\theta \mid y^{(k)}) d\theta$ | `np.trapezoid(theta_grid * f_k, theta_grid)` |
| **MAP Estimate (Posterior Mode)** | $\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta} f(\theta \mid y^{(k)})$ | `theta_grid[np.argmax(f_k)]` |
| **Posterior Variance** | $\text{Var}(\theta \mid y^{(k)}) = \mathbb{E}[\theta^2 \mid y^{(k)}] - (\hat{\theta}_{\text{Bayes}}^{(k)})^2$ | `np.trapezoid((theta_grid - theta_bayes)**2 * f_k, theta_grid)` |

---

## 5. Python Implementation Script

Below is a complete executable Python script for tracking structural health parameter estimation (e.g., remaining structural stiffness ratio $\theta \in [0.1, 1.0]$) using Plotly for visualization:

```python
import numpy as np
import scipy.stats as stats
import plotly.graph_objects as go

# 1. Simulation Setup & Ground Truth Parameters
np.random.seed(42)
theta_true = 0.62  # True structural health parameter (e.g., 62% remaining stiffness)
n_steps = 15
sigma_noise = 0.05  # Sensor measurement noise standard deviation

# 2. Bounded Discretization Grid
theta_min, theta_max = 0.1, 1.0
grid_size = 500
theta_grid = np.linspace(theta_min, theta_max, grid_size)

# Uniform prior over physical bounds
f_current = np.ones(grid_size) / (theta_max - theta_min)

# Forward Physical Model h_k(theta): Non-linear response curve
def forward_model(theta, step):
    # Example non-linear response function representing structural modal frequency
    return np.sqrt(theta) * (1.0 + 0.1 * np.sin(step))

# 3. Storage for Tracking Estimates
bayes_estimates = [np.trapezoid(theta_grid * f_current, theta_grid)]
map_estimates = [theta_grid[np.argmax(f_current)]]

# 4. Sequential Updating Loop
for k in range(1, n_steps + 1):
    # Simulate actual sensor measurement with noise
    y_true = forward_model(theta_true, k)
    y_observed = y_true + np.random.normal(0, sigma_noise)
    
    # Evaluate model predictions across grid
    h_grid = forward_model(theta_grid, k)
    
    # Gaussian Likelihood evaluation
    likelihood = np.exp(-0.5 * ((y_observed - h_grid) / sigma_noise) ** 2)
    
    # Bayesian grid update & normalization
    f_unnormalized = likelihood * f_current
    norm_constant = np.trapezoid(f_unnormalized, theta_grid)
    f_current = f_unnormalized / norm_constant
    
    # Extract running point estimates
    theta_bayes = np.trapezoid(theta_grid * f_current, theta_grid)
    theta_map = theta_grid[np.argmax(f_current)]
    
    bayes_estimates.append(theta_bayes)
    map_estimates.append(theta_map)

# 5. Interactive Visualization using Plotly
steps = list(range(n_steps + 1))
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=steps, y=bayes_estimates,
    mode='lines+markers', name='Posterior Mean (Bayes Estimate)',
    line=dict(color='blue', width=2.5)
))

fig.add_trace(go.Scatter(
    x=steps, y=map_estimates,
    mode='lines+markers', name='MAP Estimate',
    line=dict(color='green', width=2, dash='dash')
))

fig.add_hline(
    y=theta_true, line_dash="dash", line_color="red", line_width=2,
    annotation_text=f"True Health Parameter (θ = {theta_true})"
)

fig.update_layout(
    title="Bayesian Estimation for Structural Health Monitoring via Bounded Grid Updates",
    xaxis_title="Sensor Measurement Step (k)",
    yaxis_title="Estimated Structural Parameter (θ)",
    template="plotly_white"
)

fig.show()

```